# 🏀 Miki Analítica — Tracker de Bàsquet
### Versió definitiva — Maig 2026

---

## Què obtindràs?

| | Càmera mòbil | Càmera fixa |
|---|---|---|
| Cistelles / faltes / rebots per equip | ✅ | ✅ |
| Evolució del marcador | ✅ | ✅ |
| Minuts reals jugats per jugadora | ✅ | ✅ |
| Seguiment de jugadores (IDs) | ✅ | ✅ |
| **Metres recorreguts per jugadora** | ❌ | ✅ |
| **Mapa de calor de posicions** | ❌ | ✅ |
| **Mapa de tir real sobre la pista** | ❌ | ✅ |
| **Velocitat en km/h** | ❌ | ✅ |

---

## Escenaris disponibles

- **Escenari A** — Tens el vídeo + l'ID del partit FCBQ → resultat complet
- **Escenari B** — Tens el vídeo però no l'ID del partit → tracking sense play-by-play
- **Escenari C** — Tens l'ID del partit però no el vídeo → minuts reals per jugadora

---

## Instruccions pas a pas

1. **Activa la GPU**: `Entorn d'execució → Canvia el tipus → T4 GPU`
2. **Executa les cel·les en ordre**, una per una
3. **Espera ✅** a cada cel·la abans de continuar
4. **Només cal editar** la cel·la 2 (configuració) i la cel·la 6 (ancoratges de quart)
5. **El vídeo ha d'estar a Google Drive**

---

## Com gravar el vídeo per obtenir el millor resultat

- 📱 Mòbil o iPad en **trípode**, a la **graderia alta**, centrat
- 📐 Han de ser visibles les **4 cantonades de la pista** en tot moment
- 🎬 Qualitat **1080p 30fps** (no cal 4K)
- 🔋 Connecta a bateria externa — 90 min de gravació consumeix molt
- 💾 Necessites ~6-8 GB d'espai lliure


## CEL·LA 1 — Instal·la les eines
⏱ ~3 minuts. Cal executar cada sessió nova de Colab.

In [ ]:
# Fix de Pillow (evita errors de compatibilitat)
!pip install -q 'Pillow==10.4.0'
import importlib, PIL; importlib.reload(PIL)

# Instal·la les eines principals
!pip install -q ultralytics lapx opencv-python-headless yt-dlp

import torch, cv2, numpy as np, pandas as pd
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO DISPONIBLE — activa la GPU!'
print(f'✅ Instal·lació completada')
print(f'   GPU: {gpu}')
if 'NO' in gpu: print('   ⚠️  Ves a Entorn execució → Canvia tipus → T4 GPU')


## CEL·LA 2 — Configuració del partit
✏️ **Edita només aquesta cel·la cada vegada que processen un nou partit.**

In [ ]:
# ════════════════════════════════════════════════════════
# EDITA AQUÍ — les úniques línies que cal canviar
# ════════════════════════════════════════════════════════

ESCENARI       = 'A'    # 'A' = vídeo + API FCBQ (recomanat)
                        # 'B' = només vídeo (sense ID de partit)
                        # 'C' = només API (sense vídeo)

MATCH_ID       = '69ec95d4339c3d0001f523a1'  # ID del partit FCBQ
                                              # (de la URL del partit)

EQUIP_LOCAL    = 'Miki Lakers'    # nom de l'equip local
EQUIP_VISITANT = 'Mikinaikos'     # nom de l'equip visitant
NOM_SORTIDA    = 'dades_partit'   # nom base dels fitxers CSV exportats

# Ruta del vídeo a Google Drive
# Per trobar-la: panell esquerre 📁 → drive → MyDrive → clic dret → Copia la ruta
VIDEO_PATH = '/content/drive/MyDrive/basquet/partit.mp4'

# Color de samarreta de cada equip
COLOR_LOCAL    = 'clar'   # 'clar' = blanc/clar · 'fosc' = blau/negre/fosc
COLOR_VISITANT = 'fosc'

# Dimensions de la pista (FIBA estàndard — no cal canviar)
PISTA_LLARG = 28.0  # metres
PISTA_AMPLE = 15.0  # metres

# ════════════════════════════════════════════════════════
print(f'✅ Escenari {ESCENARI}: {EQUIP_LOCAL} vs {EQUIP_VISITANT}')
if ESCENARI in ['A','C']: print(f'   Match ID: {MATCH_ID}')
if ESCENARI in ['A','B']: print(f'   Vídeo: {VIDEO_PATH}')


## CEL·LA 3 — Connecta Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os

CAMERA_FIXA = False
HOMOGRAFIA  = None

if ESCENARI in ['A','B']:
    if os.path.exists(VIDEO_PATH):
        cap = cv2.VideoCapture(VIDEO_PATH)
        FPS_VIDEO    = cap.get(cv2.CAP_PROP_FPS)
        TOTAL_FRAMES = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        ret, FRAME_INICIAL = cap.read()
        FRAME_INICIAL_RGB  = cv2.cvtColor(FRAME_INICIAL, cv2.COLOR_BGR2RGB)
        cap.release()
        DURACIO_SEG = TOTAL_FRAMES / FPS_VIDEO
        mida = os.path.getsize(VIDEO_PATH)/1024/1024/1024
        print(f'✅ Vídeo: {DURACIO_SEG/60:.1f} min · {FPS_VIDEO:.0f} fps · {TOTAL_FRAMES} frames · {mida:.1f} GB')
        # Mostra el primer frame
        import matplotlib.pyplot as plt
        fig, ax = plt.subplots(figsize=(12,7))
        ax.imshow(FRAME_INICIAL_RGB); ax.axis('off')
        ax.set_title('Primer frame del vídeo — comprova que es veu la pista sencera', fontsize=12)
        plt.tight_layout(); plt.show()
    else:
        print(f'⚠️  Vídeo no trobat: {VIDEO_PATH}')
        print('   Comprova la ruta a la cel·la 2')
else:
    FRAME_INICIAL = None; FRAME_INICIAL_RGB = None
    print('Escenari C: sense vídeo.')


## CEL·LA 4 — Descarrega les dades de l'API FCBQ
Escenari B: executa igualment, la saltarà automàticament.

In [ ]:
import urllib.request, json

df_api = pd.DataFrame()
equip_noms = {}

if ESCENARI in ['A','C']:
    print('Descarregant API FCBQ...')
    url = f'https://msstats.optimalwayconsulting.com/v1/fcbq/getJsonWithMatchMoves/{MATCH_ID}?currentSeason=true'
    req = urllib.request.Request(url, headers={
        'User-Agent':'Mozilla/5.0','Accept':'application/json',
        'Referer':f'https://www.basquetcatala.cat/estadistiques/2025/{MATCH_ID}'})
    with urllib.request.urlopen(req, timeout=15) as r:
        data = json.loads(r.read())
    moves = data if isinstance(data, list) else data.get('moves',[])
    rows = []
    for m in moves:
        mn=m.get('min',0); sc=m.get('sec',0); q=m.get('period',1)
        rows.append({'temps_seg':round(((q-1)*10+mn+sc/60)*60,1),
            'quart':q,'min':mn,'sec':sc,
            'jugadora':m.get('actorName',''),
            'idEquip':str(m.get('idTeam','')),
            'accio':m.get('move',''),
            'idMove':m.get('idMove',0),
            'marcador':m.get('score','')})
    df_api = pd.DataFrame(rows)
    equips = [e for e in df_api['idEquip'].unique() if e and e!='0']
    if len(equips)>=2:
        equip_noms = {equips[0]:EQUIP_LOCAL, equips[1]:EQUIP_VISITANT}
    df_api['equip_nom'] = df_api['idEquip'].map(equip_noms).fillna('?')
    print(f'✅ {len(df_api)} jugades · Equips: {equip_noms}')
    print(f'   Substitucions: {len(df_api[df_api["idMove"].isin([112,115])])}')
else:
    print('Escenari B: sense API.')


## CEL·LA 5 — Minuts reals jugats (des de l'API)
**Escenari C**: quan acabi aquesta cel·la ja tens tot. Pots anar a la cel·la 12 per exportar.

In [ ]:
minuts_jugades = {}

if ESCENARI in ['A','C'] and not df_api.empty:
    DQ=600; intervals={}; en_pista={}
    for _,row in df_api.sort_values('temps_seg').iterrows():
        jug=row['jugadora']; t=row['temps_seg']; idm=row['idMove']; q=row['quart']
        if not jug: continue
        if idm==112: en_pista[jug]=t
        elif idm==115:
            ini=en_pista.pop(jug,(q-1)*DQ)
            intervals.setdefault(jug,[]).append((ini,t))
        elif idm==116:
            fi=q*DQ
            for j,ti in list(en_pista.items()): intervals.setdefault(j,[]).append((ti,fi))
            en_pista={}
    for jug,ivs in intervals.items():
        minuts_jugades[jug]=round(sum(f-i for i,f in ivs)/60,1)
    df_min=pd.DataFrame([{'Jugadora':j,'Minuts':m,
        'Equip':df_api[df_api['jugadora']==j]['equip_nom'].iloc[0]
        if not df_api[df_api['jugadora']==j].empty else '?'}
        for j,m in sorted(minuts_jugades.items(),key=lambda x:-x[1])])
    print('✅ Minuts reals per jugadora:')
    print(df_min.to_string(index=False))
else:
    print('Sense API: minuts calculats des del vídeo.')


## CEL·LA 6 — Sincronització temps de joc ↔ vídeo
✏️ **Edita els 4 valors.** Obre el vídeo al VLC, busca el moment on l'àrbitre fa el salt
de cada quart i apunta el segon del vídeo.

In [ ]:
# ════════════════════════════════════════════════════════
# EDITA AQUÍ: segon del vídeo on comença cada quart
# Exemple: si Q1 comença al minut 0:45 del vídeo → posa 45.0
ANCORATGES_VIDEO = {
    1: 45.0,    # Q1
    2: 680.0,   # Q2
    3: 1290.0,  # Q3
    4: 1870.0,  # Q4
}
# Si el vídeo és curt i no inclou tots els quarts,
# posa 9999.0 per als quarts que no existeixen
# ════════════════════════════════════════════════════════

DQ_JOC=600; AJUSTOS_TL={}

def joc_a_video(t_joc, quart):
    if quart not in ANCORATGES_VIDEO: return t_joc
    prop=max(0,min(1,(t_joc-(quart-1)*DQ_JOC)/DQ_JOC))
    dur=(ANCORATGES_VIDEO[quart+1]-ANCORATGES_VIDEO[quart] if quart<4 else DQ_JOC*1.5)
    tv=ANCORATGES_VIDEO[quart]+prop*dur
    return round(tv,2)

print('✅ Sincronització configurada')
for q in sorted(ANCORATGES_VIDEO)[:-1]:
    at=ANCORATGES_VIDEO[q+1]-ANCORATGES_VIDEO[q]-DQ_JOC
    if ANCORATGES_VIDEO[q+1] < 9999:
        print(f'  Q{q}: ~{at:.0f}s d\'aturades estimades ({at/60:.1f} min)')


## CEL·LA 7 — Detecció automàtica del tipus de càmera
Detecta si la pista és visible sencera. Si és càmera fixa, activa totes les funcionalitats.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, matplotlib.patches as patches

def pixels_a_metres(px, py, H=None):
    h = H if H is not None else HOMOGRAFIA
    if h is None: return None, None
    p=np.array([[[float(px),float(py)]]],dtype=np.float32)
    r=cv2.perspectiveTransform(p,h)
    return round(float(np.clip(r[0][0][0],0,PISTA_LLARG)),2),\
           round(float(np.clip(r[0][0][1],0,PISTA_AMPLE)),2)

if ESCENARI in ['A','B'] and FRAME_INICIAL is not None:
    # Intenta detectar les cantonades automàticament
    gray=cv2.cvtColor(FRAME_INICIAL,cv2.COLOR_BGR2GRAY)
    edges=cv2.Canny(cv2.GaussianBlur(gray,(5,5),0),50,150)
    lines=cv2.HoughLinesP(edges,1,np.pi/180,threshold=100,
                          minLineLength=FRAME_INICIAL.shape[1]//4,maxLineGap=20)
    cantonades=None
    if lines is not None and len(lines)>=4:
        horitz=[l[0] for l in lines if abs(l[0][1]-l[0][3])<20]
        vertic=[l[0] for l in lines if abs(l[0][0]-l[0][2])<20]
        if len(horitz)>=2 and len(vertic)>=2:
            hy=sorted([min(l[1],l[3]) for l in horitz])
            vx=sorted([min(l[0],l[2]) for l in vertic])
            y_t,y_b=hy[0],hy[-1]; x_l,x_r=vx[0],vx[-1]
            h,w=FRAME_INICIAL.shape[:2]
            if (x_r-x_l)>=w*0.5 and (y_b-y_t)>=h*0.4:
                cantonades=np.float32([[x_l,y_t],[x_r,y_t],[x_r,y_b],[x_l,y_b]])
    if cantonades is not None:
        CAMERA_FIXA=True
        HOMOGRAFIA,_=cv2.findHomography(cantonades,
            np.float32([[0,PISTA_AMPLE],[PISTA_LLARG,PISTA_AMPLE],[PISTA_LLARG,0],[0,0]]))
        print('✅ CÀMERA FIXA detectada automàticament!')
        print('   Activades: metres recorreguts · mapa de tir · mapa de calor')
    else:
        CAMERA_FIXA=False
        print('ℹ️  Càmera mòbil o pista no visible sencera.')
        print('   Si és càmera fixa però no detecta, executa la cel·la 7b.')
    # Mostra resultat
    fig,ax=plt.subplots(figsize=(12,7))
    ax.imshow(FRAME_INICIAL_RGB); ax.axis('off')
    if CAMERA_FIXA and cantonades is not None:
        cv2_c=np.vstack([cantonades,cantonades[0]])
        ax.plot(cv2_c[:,0],cv2_c[:,1],'g-',lw=2)
        for c,n in zip(cantonades,['Sup.Esq','Sup.Dret','Inf.Dret','Inf.Esq']):
            ax.plot(c[0],c[1],'gs',ms=10)
            ax.text(c[0]+5,c[1]-5,n,color='lime',fontsize=9,
                    bbox=dict(facecolor='black',alpha=0.6,edgecolor='none',pad=2))
    color_t='#16a34a' if CAMERA_FIXA else '#d97706'
    ax.set_title(f'{"✅ CÀMERA FIXA" if CAMERA_FIXA else "ℹ️  CÀMERA MÒBIL"}',
                fontsize=13,color=color_t,fontweight='bold')
    plt.tight_layout(); plt.show()
else:
    CAMERA_FIXA=False
    print('Escenari C: sense vídeo.')


## CEL·LA 7b — Cantonades manuals (opcional)
⚠️ Executa NOMÉS si la cel·la 7 ha dit 'càmera mòbil' però el vídeo SÍ és de càmera fixa.
Mira el primer frame i apunta les coordenades en píxels de les 4 cantonades de la pista.

In [ ]:
# ════════════════════════════════════════════════════════
# EDITA AQUÍ: coordenades en píxels dels 4 cantons
#
#  CANT_1 ───────────── CANT_2
#    │        PISTA        │
#  CANT_4 ───────────── CANT_3
#
CANTONADES_MANUAL = np.float32([
    [95,   60],   # Cantonada superior esquerra
    [1185, 60],   # Cantonada superior dreta
    [1185, 660],  # Cantonada inferior dreta
    [95,   660],  # Cantonada inferior esquerra
])
# ════════════════════════════════════════════════════════

CAMERA_FIXA = True
HOMOGRAFIA, _ = cv2.findHomography(
    CANTONADES_MANUAL,
    np.float32([[0,PISTA_AMPLE],[PISTA_LLARG,PISTA_AMPLE],[PISTA_LLARG,0],[0,0]]))
print('✅ Cantonades manuals aplicades')
print(f'  Test cantonada inf.esq: {pixels_a_metres(*CANTONADES_MANUAL[3])} (ha de ser 0.0, 0.0)')
print(f'  Test cantonada sup.dret: {pixels_a_metres(*CANTONADES_MANUAL[1])} (ha de ser {PISTA_LLARG}, {PISTA_AMPLE})')


## CEL·LA 8 — Carrega el model YOLOv8
⏱ ~1 minut

In [ ]:
from ultralytics import YOLO
import torch, time

model_tracking = YOLO('yolov8n.pt')
print(f'✅ Model YOLOv8 carregat')
print(f'   GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (lent)"}')
print(f'   Mode: {"CÀMERA FIXA" if CAMERA_FIXA else "CÀMERA MÒBIL"}')
print()
print('Funcionalitats disponibles:')
print(f'  ✅ Tracking de jugadores')
print(f'  {"✅" if not df_api.empty else "❌"} Estadístiques d\'accions (API)')
print(f'  {"✅" if not df_api.empty else "❌"} Minuts reals jugats')
print(f'  {"✅" if CAMERA_FIXA else "❌"} Metres recorreguts per jugadora')
print(f'  {"✅" if CAMERA_FIXA else "❌"} Mapa de tir real')
print(f'  {"✅" if CAMERA_FIXA else "❌"} Mapa de calor de posicions')


## CEL·LA 9 — Processa el vídeo
⏱ 15-30 minuts per a un partit complet. Deixa-ho córrer.

In [ ]:
from tqdm import tqdm

if ESCENARI not in ['A','B'] or FRAME_INICIAL is None:
    print('Escenari C: sense vídeo. Salta a la cel·la 11.')
else:
    cap     = cv2.VideoCapture(VIDEO_PATH)
    fps_vid = cap.get(cv2.CAP_PROP_FPS)
    total   = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    INTERVAL = max(1, int(fps_vid/5))  # 5 frames per segon
    print(f'Processant {total//INTERVAL} frames ({total//INTERVAL/300:.0f} min estimats)...')

    resultats=[]; frame_idx=0; t0=time.time()

    with tqdm(total=total//INTERVAL) as pbar:
        while cap.isOpened():
            ret,frame=cap.read()
            if not ret: break
            if frame_idx%INTERVAL==0:
                temps_seg=frame_idx/fps_vid
                results=model_tracking.track(frame,persist=True,
                    tracker='bytetrack.yaml',classes=[0],conf=0.4,verbose=False)
                if results[0].boxes is not None and results[0].boxes.id is not None:
                    boxes=results[0].boxes.xyxy.cpu().numpy()
                    ids=results[0].boxes.id.cpu().numpy().astype(int)
                    for box,tid in zip(boxes,ids):
                        cx_px=(box[0]+box[2])/2; cy_px=box[3]
                        cx_m=cy_m=None
                        if CAMERA_FIXA:
                            cx_m,cy_m=pixels_a_metres(cx_px,cy_px)
                        roi=frame[int(box[1]):int(box[3]),int(box[0]):int(box[2])]
                        llum=float(roi.mean()) if roi.size>0 else 128
                        eq=EQUIP_LOCAL if ((COLOR_LOCAL=='clar' and llum>128) or
                                           (COLOR_LOCAL=='fosc' and llum<=128)) else EQUIP_VISITANT
                        resultats.append({'temps_seg':round(temps_seg,2),'frame':frame_idx,
                            'track_id':int(tid),'equip':eq,
                            'x_pixels':round(cx_px,1),'y_pixels':round(cy_px,1),
                            'x_metres':cx_m,'y_metres':cy_m})
                pbar.update(1)
            frame_idx+=1
    cap.release()

    df_tracking=pd.DataFrame(resultats)
    df_tracking['track_id']=df_tracking['track_id'].astype(str)
    df_tracking['jugadora']=df_tracking['track_id'].apply(lambda x: f'Jugadora_{x}')

    if CAMERA_FIXA and not df_tracking.empty:
        df_tracking=df_tracking.sort_values(['track_id','temps_seg'])
        df_tracking['dx']=df_tracking.groupby('track_id')['x_metres'].diff().fillna(0)
        df_tracking['dy']=df_tracking.groupby('track_id')['y_metres'].diff().fillna(0)
        df_tracking['dist_frame']=np.sqrt(df_tracking['dx']**2+df_tracking['dy']**2)
        df_tracking.loc[df_tracking['dist_frame']>5,'dist_frame']=0
    else:
        df_tracking['dist_frame']=0

    elapsed=time.time()-t0
    print(f'\n✅ Completat en {elapsed/60:.1f} min')
    print(f'   {len(df_tracking)} registres · {df_tracking["track_id"].nunique()} persones detectades')


## CEL·LA 10 — Creuament accions API + posicions del vídeo

In [ ]:
df_accions = pd.DataFrame()

if not df_api.empty and 'df_tracking' in dir() and not df_tracking.empty:
    print('Creuant API amb posicions del vídeo...')
    accions_geo=[]
    for _,row in df_api.iterrows():
        accio=row['accio']; jugadora=row['jugadora']
        if not accio: continue
        t_vid=joc_a_video(row['temps_seg'],row['quart'])
        eq_nom=equip_noms.get(row['idEquip'],'?')
        x_m=y_m=None
        if CAMERA_FIXA:
            df_eq_frame=df_tracking[
                (df_tracking['equip']==eq_nom)&
                ((df_tracking['temps_seg']-t_vid).abs()<=3)]
            if not df_eq_frame.empty:
                idx_p=(df_eq_frame['temps_seg']-t_vid).abs().idxmin()
                fila=df_eq_frame.loc[idx_p]
                if fila['x_metres'] is not None:
                    x_m,y_m=fila['x_metres'],fila['y_metres']
        tipus='altre'
        if any(c in accio for c in ['Cistella de 2','Cistella de 3','Cistella de 1']): tipus='cistella'
        elif 'Intent fallat' in accio: tipus='tir_fallat'
        elif 'falta' in accio.lower(): tipus='falta'
        elif 'Rebot' in accio: tipus='rebot'
        accions_geo.append({'temps_joc':row['temps_seg'],'temps_video':t_vid,
            'quart':row['quart'],'jugadora':jugadora,'equip':eq_nom,
            'accio':accio,'tipus':tipus,'marcador':row['marcador'],
            'x_metres':round(x_m,2) if x_m else None,
            'y_metres':round(y_m,2) if y_m else None,
            'geolocalitzat':x_m is not None})
    df_accions=pd.DataFrame(accions_geo)
    geo=df_accions['geolocalitzat'].sum(); total_a=len(df_accions)
    print(f'✅ {total_a} accions analitzades')
    if CAMERA_FIXA: print(f'   {geo}/{total_a} geolocalitzades ({geo/total_a*100:.0f}%)')
    print()
    print(df_accions.groupby(['equip','tipus']).size().unstack(fill_value=0))
elif not df_api.empty:
    # Sense vídeo: estadístiques de l'API directament
    accions_geo=[]
    for _,row in df_api.iterrows():
        if not row['accio']: continue
        tipus='altre'
        if any(c in row['accio'] for c in ['Cistella de 2','Cistella de 3','Cistella de 1']): tipus='cistella'
        elif 'Intent fallat' in row['accio']: tipus='tir_fallat'
        elif 'falta' in row['accio'].lower(): tipus='falta'
        elif 'Rebot' in row['accio']: tipus='rebot'
        accions_geo.append({'temps_joc':row['temps_seg'],'quart':row['quart'],
            'jugadora':row['jugadora'],'equip':row['equip_nom'],
            'accio':row['accio'],'tipus':tipus,'marcador':row['marcador'],
            'x_metres':None,'y_metres':None,'geolocalitzat':False})
    df_accions=pd.DataFrame(accions_geo)
    print(f'✅ {len(df_accions)} accions de l\'API')
    print(df_accions.groupby(['equip','tipus']).size().unstack(fill_value=0))


## CEL·LA 11 — Genera els mapes i gràfics visuals

In [ ]:
import matplotlib.pyplot as plt, matplotlib.patches as patches

def dibuixa_pista(ax):
    ax.set_facecolor('#f0f4f8'); ax.set_aspect('equal')
    ax.set_xlim(-0.5,PISTA_LLARG+0.5); ax.set_ylim(-0.5,PISTA_AMPLE+0.5)
    ax.add_patch(patches.Rectangle((0,0),PISTA_LLARG,PISTA_AMPLE,fill=False,edgecolor='#374151',lw=2))
    ax.add_patch(patches.Rectangle((0,PISTA_AMPLE/2-3),5.8,6,facecolor='#dbeafe',edgecolor='#374151',lw=1))
    ax.add_patch(patches.Rectangle((PISTA_LLARG-5.8,PISTA_AMPLE/2-3),5.8,6,facecolor='#dbeafe',edgecolor='#374151',lw=1))
    ax.add_patch(patches.Arc((0,PISTA_AMPLE/2),27.5,27.5,angle=0,theta1=-90,theta2=90,color='#374151',lw=1.5))
    ax.add_patch(patches.Arc((PISTA_LLARG,PISTA_AMPLE/2),27.5,27.5,angle=0,theta1=90,theta2=270,color='#374151',lw=1.5))
    ax.plot(1.575,PISTA_AMPLE/2,'o',color='#374151',ms=7)
    ax.plot(PISTA_LLARG-1.575,PISTA_AMPLE/2,'o',color='#374151',ms=7)
    ax.axvline(PISTA_LLARG/2,color='#374151',lw=1.5)

equips_vid=[e for e in df_accions['equip'].unique() if e and e!='?'] if not df_accions.empty else [EQUIP_LOCAL, EQUIP_VISITANT]

# ── Gràfic estadístiques per equip ─────────────────────────────────────
if not df_accions.empty:
    fig,axes=plt.subplots(1,2,figsize=(14,5))
    fig.suptitle(f'{EQUIP_LOCAL} vs {EQUIP_VISITANT}',fontsize=13,fontweight='bold')
    for ax,eq,color in [(axes[0],equips_vid[0] if equips_vid else EQUIP_LOCAL,'#185FA5'),
                        (axes[1],equips_vid[1] if len(equips_vid)>1 else EQUIP_VISITANT,'#993C1D')]:
        df_e=df_accions[df_accions['equip']==eq]
        cats=['cistella','tir_fallat','falta','rebot']
        etiq=['Cistelles','Tirs fallats','Faltes','Rebots']
        vals=[int((df_e['tipus']==c).sum()) for c in cats]
        bars=ax.bar(etiq,vals,color=color,alpha=0.8,edgecolor='white',lw=1.5)
        ax.bar_label(bars,padding=3,fontsize=11,fontweight='bold')
        ax.set_title(eq,fontsize=12,color=color,fontweight='bold')
        ax.set_facecolor('#f9fafb'); ax.spines[['top','right']].set_visible(False)
        ax.set_ylim(0,max(vals)*1.25 if max(vals)>0 else 10)
    plt.tight_layout()
    plt.savefig('estadistiques_partit.png',dpi=150,bbox_inches='tight'); plt.show()
    print('✅ Gràfic estadístiques generat')

# ── Mapa de tir real (només càmera fixa) ────────────────────────────────
if CAMERA_FIXA and not df_accions.empty and df_accions['geolocalitzat'].any():
    df_tirs=df_accions[df_accions['tipus'].isin(['cistella','tir_fallat'])&df_accions['geolocalitzat']]
    if not df_tirs.empty:
        fig,axes=plt.subplots(1,2,figsize=(16,7))
        fig.suptitle('Mapa de tir REAL — coordenades sobre la pista',fontsize=14,fontweight='bold')
        for ax,eq,ca,cf in [(axes[0],equips_vid[0] if equips_vid else EQUIP_LOCAL,'#185FA5','#93c5fd'),
                            (axes[1],equips_vid[1] if len(equips_vid)>1 else EQUIP_VISITANT,'#993C1D','#fca5a5')]:
            dibuixa_pista(ax)
            df_e=df_tirs[df_tirs['equip']==eq]
            cist=df_e[df_e['tipus']=='cistella']; fall=df_e[df_e['tipus']=='tir_fallat']
            ax.scatter(fall['x_metres'],fall['y_metres'],c=cf,s=80,marker='x',lw=2,alpha=0.7,label=f'Fallat ({len(fall)})',zorder=3)
            ax.scatter(cist['x_metres'],cist['y_metres'],c=ca,s=100,marker='o',edgecolors='white',lw=1.5,alpha=0.85,label=f'Cistella ({len(cist)})',zorder=4)
            ef=round(len(cist)/(len(cist)+len(fall))*100) if (len(cist)+len(fall))>0 else 0
            ax.set_title(f'{eq} — {ef}% eficiència',fontsize=12,color=ca,fontweight='bold')
            ax.legend(fontsize=10); ax.set_xlabel('Metres'); ax.set_ylabel('Metres')
        plt.tight_layout()
        plt.savefig('mapa_tir_real.png',dpi=150,bbox_inches='tight'); plt.show()
        print('✅ Mapa de tir real generat!')

# ── Mapa de calor (només càmera fixa) ───────────────────────────────────
if CAMERA_FIXA and 'df_tracking' in dir() and not df_tracking.empty:
    df_pos=df_tracking.dropna(subset=['x_metres','y_metres'])
    if not df_pos.empty:
        fig,axes=plt.subplots(1,2,figsize=(16,7))
        fig.suptitle('Mapa de calor de posicions',fontsize=14,fontweight='bold')
        for ax,eq,cmap,color in [(axes[0],EQUIP_LOCAL,'Blues','#185FA5'),(axes[1],EQUIP_VISITANT,'Reds','#993C1D')]:
            dibuixa_pista(ax)
            df_e=df_pos[df_pos['equip']==eq]
            if not df_e.empty:
                h=ax.hist2d(df_e['x_metres'],df_e['y_metres'],bins=[28,15],range=[[0,28],[0,15]],cmap=cmap,alpha=0.65)
                plt.colorbar(h[3],ax=ax,label='Presència')
            ax.set_title(eq,fontsize=12,color=color,fontweight='bold')
        plt.tight_layout()
        plt.savefig('mapa_calor.png',dpi=150,bbox_inches='tight'); plt.show()
        print('✅ Mapa de calor generat!')


## CEL·LA 12 — Estadístiques i exporta els CSV per a Miki Analítica

In [ ]:
from google.colab import files
import os

# ── Estadístiques per jugadora ─────────────────────────────────────────
if 'df_tracking' in dir() and not df_tracking.empty:
    df_tracking['track_id']=df_tracking['track_id'].astype(str)
    resum=df_tracking.groupby(['track_id','equip']).agg(
        aparicions=('temps_seg','count'),
        temps_min=('temps_seg','min'),
        temps_max=('temps_seg','max'),
    ).reset_index()
    resum['minuts_visibles']=((resum['temps_max']-resum['temps_min'])/60).round(1)
    if CAMERA_FIXA:
        dist=df_tracking.groupby('track_id')['dist_frame'].sum().reset_index()
        dist.columns=['track_id','metres_total']
        dist['metres_total']=dist['metres_total'].round(0)
        resum=resum.merge(dist,on='track_id',how='left')
        resum['m_per_min']=(resum['metres_total']/resum['minuts_visibles'].replace(0,1)).round(1)
    print('📊 Estadístiques per ID de tracking:')
    cols=['track_id','equip','minuts_visibles']
    if CAMERA_FIXA and 'metres_total' in resum.columns: cols+=['metres_total','m_per_min']
    print(resum[cols].sort_values('minuts_visibles',ascending=False).head(15).to_string(index=False))

# Minuts de l'API si disponibles
if minuts_jugades:
    print()
    print('📊 Minuts reals (API):')
    for j,m in sorted(minuts_jugades.items(),key=lambda x:-x[1]):
        print(f'   {j:30s} {m:.1f} min')

# ── Exporta fitxers ─────────────────────────────────────────────────────
print()
print('Descarregant fitxers per a Miki Analítica...')
exports=[]

if 'df_tracking' in dir() and not df_tracking.empty:
    f1=f'{NOM_SORTIDA}_tracking.csv'
    cols_exp=['temps_seg','frame','track_id','jugadora','equip','x_pixels','y_pixels']
    if CAMERA_FIXA: cols_exp+=['x_metres','y_metres','dist_frame']
    df_tracking[cols_exp].to_csv(f1,index=False); exports.append(f1)

if 'resum' in dir():
    f2=f'{NOM_SORTIDA}_resum.csv'
    resum.to_csv(f2,index=False); exports.append(f2)

if not df_accions.empty:
    f3=f'{NOM_SORTIDA}_accions.csv'
    df_accions.to_csv(f3,index=False); exports.append(f3)

for img in ['estadistiques_partit.png','mapa_tir_real.png','mapa_calor.png']:
    if os.path.exists(img): exports.append(img)

for f in exports:
    try: files.download(f); print(f'  ✅ {f}')
    except: print(f'  ⚠️  {f} no trobat')

print()
print('🏀 Fet! Importa els CSV a la pestanya 🎬 Vídeo de Miki Analítica.')
print(f'   Mode: {"CÀMERA FIXA" if CAMERA_FIXA else "CÀMERA MÒBIL"}')
if CAMERA_FIXA:
    print('   Has obtingut: tracking + metres + mapa de tir + mapa de calor')
else:
    print('   Has obtingut: tracking + estadístiques d\'accions')
    print('   Per obtenir metres i mapes: grava amb càmera fixa a la graderia alta')
